In [12]:
import pandas as pd

In [13]:
import pandas as pd
import numpy as np
from rdkit import Chem
from rdkit.Chem import AllChem
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report

In [14]:
import pubchempy as pcp  # Import the new library
from tqdm import tqdm  # A helper to show progress bars

In [16]:
df = pd.read_csv(r"C:\Users\ZIAD\Downloads\TWOSIDES.csv\TWOSIDES.csv")
print(df.head())

C:\Users\ZIAD\AppData\Local\Temp\ipykernel_3380\4095106583.py:1: DtypeWarning: Columns (0,2,4,6,7,8,9,10,11,12) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(r"C:\Users\ZIAD\Downloads\TWOSIDES.csv\TWOSIDES.csv")


  drug_1_rxnorn_id       drug_1_concept_name drug_2_rxnorm_id  \
0            10355                 Temazepam           136411   
1             1808                Bumetanide             7824   
2           221147  POLYETHYLENE GLYCOL 3350             5521   
3            10324                 Tamoxifen             8640   
4            10355                 Temazepam           136411   

  drug_2_concept_name condition_meddra_id condition_concept_name   A    B   C  \
0          sildenafil            10003239             Arthralgia   7  149  24   
1            Oxytocin            10003239             Arthralgia   1   13   2   
2  Hydroxychloroquine            10003239             Arthralgia   6  103  20   
3          Prednisone            10012735              Diarrhoea  18  123  35   
4          sildenafil            10012735              Diarrhoea   2  154  37   

      D       PRR PRR_error mean_reporting_frequency  
0  1536   2.91667  0.421275                0.0448718  
1   138     

In [17]:
print("--- Original Data ---")
print(df)
print("-" * 30)

--- Original Data ---
         drug_1_rxnorn_id       drug_1_concept_name drug_2_rxnorm_id  \
0                   10355                 Temazepam           136411   
1                    1808                Bumetanide             7824   
2                  221147  POLYETHYLENE GLYCOL 3350             5521   
3                   10324                 Tamoxifen             8640   
4                   10355                 Temazepam           136411   
...                   ...                       ...              ...   
42920386             6142                Ketoprofen            88249   
42920387             6142                Ketoprofen            88249   
42920388             6142                Ketoprofen            88249   
42920389             6142                Ketoprofen            88249   
42920390             6142                Ketoprofen            88249   

         drug_2_concept_name condition_meddra_id  \
0                 sildenafil            10003239   
1        

In [18]:
print("My CSV columns are:", df.columns)

My CSV columns are: Index(['drug_1_rxnorn_id', 'drug_1_concept_name', 'drug_2_rxnorm_id',
       'drug_2_concept_name', 'condition_meddra_id', 'condition_concept_name',
       'A', 'B', 'C', 'D', 'PRR', 'PRR_error', 'mean_reporting_frequency'],
      dtype='object')


In [19]:
# --- 2. NEW: Fetch SMILES Strings ---

# Get all unique drug names from both columns to avoid duplicate lookups
unique_drug_names = pd.concat([
    df['drug_1_concept_name'], 
    df['drug_2_concept_name']
]).unique()

# Create a mapping dictionary {drug_name: smiles}
drug_name_to_smiles = {}

# Use tqdm for a progress bar
for name in tqdm(unique_drug_names, desc="Fetching SMILES"):
    try:
        # Search PubChem by name
        compounds = pcp.get_compounds(name, 'name')
        if compounds:
            # Get the first match's SMILES string
            drug_name_to_smiles[name] = compounds[0].canonical_smiles
        else:
            drug_name_to_smiles[name] = None  # No match found
    except Exception as e:
        # Handle API errors, etc.
        print(f"Error fetching {name}: {e}")
        drug_name_to_smiles[name] = None

print("SMILES lookup complete.")

# Map the SMILES strings back to your dataframe
df['drug1_smiles'] = df['drug_1_concept_name'].map(drug_name_to_smiles)
df['drug2_smiles'] = df['drug_2_concept_name'].map(drug_name_to_smiles)

Fetching SMILES:   0%|          | 0/1918 [00:00<?, ?it/s]C:\Users\ZIAD\AppData\Local\Temp\ipykernel_3380\781721680.py:19: PubChemPyDeprecationWarning: canonical_smiles is deprecated: Use connectivity_smiles instead
  drug_name_to_smiles[name] = compounds[0].canonical_smiles
Fetching SMILES: 100%|██████████| 1918/1918 [20:26<00:00,  1.56it/s]


SMILES lookup complete.


Original rows: 42920391, Rows with SMILES: 38798960


In [20]:
# --- 3. Clean Data ---
# Drop any rows where we couldn't find a SMILES string
df_clean = df.dropna(subset=['drug1_smiles', 'drug2_smiles']).copy()
print(f"Original rows: {len(df)}, Rows with SMILES: {len(df_clean)}")
print("-" * 30)

Original rows: 42920391, Rows with SMILES: 38798960
------------------------------


 Feature Engineering: SMILES to Fingerprints

 """
    Converts a SMILES string into a Morgan fingerprint.
    """

In [27]:
def smiles_to_fingerprint(smiles_string, n_bits=2048):
    
    try:
        mol = Chem.MolFromSmiles(smiles_string)
        if mol is None:
            # Handle invalid SMILES
            return np.zeros(n_bits, dtype=int)
        
        # Generate Morgan fingerprint
        fp = AllChem.GetMorganFingerprintAsBitVect(mol, 2, nBits=n_bits)
        return np.array(fp, dtype=int)
    except Exception as e:
        print(f"Error processing SMILES {smiles_string}: {e}")
        return np.zeros(n_bits, dtype=int)

In [ ]:
# --- 5. Create Feature Matrix (X) and Target Vector (y) ---

fp_size = 1024
print(f"Generating {fp_size}-bit fingerprints...")

# Apply the function to each row on our cleaned dataframe
df_clean['fp1'] = df_clean['drug1_smiles'].apply(lambda x: smiles_to_fingerprint(x, fp_size))
df_clean['fp2'] = df_clean['drug2_smiles'].apply(lambda x: smiles_to_fingerprint(x, fp_size))

# Concatenate the two fingerprints
X_list = [np.concatenate((fp1, fp2)) for fp1, fp2 in zip(df_clean['fp1'], df_clean['fp2'])]
X = np.array(X_list)

y = df_clean['condition_concept_name'].values

print(f"Feature matrix 'X' shape: {X.shape}")
print(f"Target vector 'y' shape: {y.shape}")
print("-" * 30)